In [11]:
import json
import pandas as pd
import os

_HERE = os.path.dirname(os.path.abspath('.'))
_HERE = '/mnt/SSD4/kartik/global-drug-pred/agentic'
_ROOT = os.path.dirname(_HERE)

with open(f'{_HERE}/split_results.json') as f:
    split_results = json.load(f)
with open(f'{_HERE}/clean_output.json') as f:
    clean_output = json.load(f)

df = pd.read_csv(
    f'{_ROOT}/data/combined_dataset.csv',
    sep=';', engine='python', quotechar='"', doublequote=True, escapechar='\\',
)
df = df.drop_duplicates(subset=['Name: ', 'Date of visit(0 months)', 'Date of visit(6 months)', 'Date of visit(12 months)'])

def get_pid(row):
    rid  = str(row.get('Record ID', '')).strip() if pd.notna(row.get('Record ID')) else ''
    name = str(row.get('Name: ', '')).strip()     if pd.notna(row.get('Name: '))    else ''
    return f'{rid}_{name}' if rid and name else name or rid

pid_to_row = {get_pid(row): row for _, row in df.iterrows()}

VISIT_LABELS = {
    'Visit_1': 'Visit 1 (0 months)',
    'Visit_2': 'Visit 2 (6 months)',
    'Visit_3': 'Visit 3 (12 months)',
}

def safe_get(row, col):
    val = row.get(col, '')
    s = str(val).strip()
    return s if s and s.lower() != 'nan' else ''

def build_patient_header(row):
    age   = safe_get(row, 'Age')
    sex   = safe_get(row, 'Sex:')
    dx    = safe_get(row, 'Seizure Diagnosis')
    onset = safe_get(row, 'Age of onset of seizure')
    dur   = safe_get(row, 'Duration of Seizure')
    parts = []
    demo = ' | '.join(x for x in [
        f'Age: {age}' if age else '',
        f'Sex: {sex}' if sex else '',
        f'Diagnosis: {dx}' if dx else '',
    ] if x)
    if demo:
        parts.append(demo)
    detail = ' | '.join(x for x in [
        f'Seizure onset: {onset}' if onset else '',
        f'Seizure duration: {dur}' if dur else '',
    ] if x)
    if detail:
        parts.append(detail)
    return '\n'.join(parts)

def build_visit_block(label, input_text, prescription=None):
    lines = [f'[{label} - Clinical Notes]']
    lines.append(input_text.strip() if input_text.strip() else '(no clinical notes recorded)')
    if prescription is not None:
        lines.append(f'\n[{label} - Prescription]')
        lines.append(prescription.strip() if prescription.strip() else '(no prescription recorded)')
    return '\n'.join(lines)

def build_pred_input(pid, visit_name):
    """Print the full prediction input for a given patient and visit."""
    visit_order = ['Visit_1', 'Visit_2', 'Visit_3']
    if visit_name not in visit_order:
        print(f'visit_name must be one of {visit_order}')
        return
    if pid not in split_results:
        print(f'Patient "{pid}" not found. Available example IDs:')
        print(list(split_results.keys())[:5])
        return

    current_idx = visit_order.index(visit_name)
    row = pid_to_row.get(pid)
    header = build_patient_header(row)

    blocks = []
    if header:
        blocks.append(header)
        blocks.append('')
    
    for prev in visit_order[:current_idx]:
        label = VISIT_LABELS[prev]
        inp  = split_results[pid].get(prev, {}).get('input_text', '')
        pres = clean_output.get(pid, {}).get(prev, '')
        blocks.append(build_visit_block(label, inp, prescription=pres))
        blocks.append('')

    label = VISIT_LABELS[visit_name]
    inp = split_results[pid].get(visit_name, {}).get('input_text', '')
    blocks.append(build_visit_block(label, inp, prescription=None))

    result = '\n'.join(blocks).strip()
    print(result)

print('Loaded. Available patients:', len(split_results))
print('Example IDs:', list(split_results.keys())[:5])

Loaded. Available patients: 279
Example IDs: ['1_Nanyonga Aisha', '2_Mwania Sheldon', '5_Nalumu Martha', '8_Tisma Natabi', '10_Muduku Matthew']


In [15]:
build_pred_input('324_Matovu Albert', 'Visit_2')

Age: 2 year old | Sex: M | Diagnosis: Focal Epilepsy
Seizure onset: Not mentioned | Seizure duration: Unknown

[Visit 1 (0 months) - Clinical Notes]
Visit 1 (Initial Visit - 0 months):
Visit Date: 2018-03-12 00:00:00
Date of Birth: 

History of Presenting Illness: This was the first available page in the file.    Date of visit: 13/12/2018  Weight: 8 kg    Reviewed a 3-year 1/12 months old male with CP (quadriplegic) on Baclofen. Has no seizures. Today reports cough and flu but no temperatures for 3 days now.    O/E: JACCOLD⁰, has a squint    CNS: Alert, PEARL, stiffness of both upper limb and lower limb    RS: Not in distress, normal and equal air entry bilaterally, chest is clear  Review of Systems: Unremarkable    
Detailed description of seizure history: Date of visit: 13/12/2018  Reviewed a 3-year 1/12 months old male with CP (quadriplegic) on Baclofen. Has no seizures. Today reports cough and flu but no temperatures for 3 days now.    Noted in the file on the subsequent visit: CP,